# Day 3b - Link Prediction

Link prediction asks: given two nodes, should there be an edge between them? In this practical, each row in a dataframe is one possible pair of nodes. We build features for those pairs, train simple models, and compare how well they find hidden edges.

We use two datasets: a Twitter mention network and a yeast protein-protein interaction (PPI) network.

## Learning objectives
By the end of this notebook you will be able to:
1. Explain link prediction as binary classification on node pairs.
2. Inspect the train graph and the held-out test dataframe.
3. Use graph-tool link-prediction APIs for local heuristics.
4. Build path-based scores, Katz scores, and node embeddings.
5. Train and compare predictors using AUC.
6. Train a small GraphSAGE model with PyTorch Geometric.
7. Inspect top predictions and compare which pairs different methods rank highly.

## Teaching rhythm
Each method is complete enough to run immediately after the short lecture block. First run the baseline exactly as written. Then make one small change, rerun the affected cell, and compare the result.

There is intentionally no separate student and solution copy. The notebook shows the method, then asks you to modify it.

## How to work through each method
For every method in this notebook:

1. Run the provided baseline cell once.
2. Write down the AUC and inspect the 10 whiteboard-pair predictions.
3. Make the specific change listed in the **Change this** instruction for that block.
4. Rerun the changed cell, plus the summary cells if the feature table changed.
5. Record one sentence explaining why the score or ranking changed.

Use this table in your notes.

| Block | Baseline AUC | What I changed | New AUC | One-sentence interpretation |
|---|---:|---|---:|---|
| Local heuristics | | | | |
| Paths / Katz | | | | |
| SVD / node2vec | | | | |
| GraphSAGE | | | | |

## Notebook map
- **Introduction**: data, prediction target, candidate-pair dataframe, and evaluation helper
- **Part 1**: Local similarity heuristics
- **Part 2**: Paths and Katz
- **Part 3**: Node embeddings (SVD, node2vec)
- **Part 4**: Graph Neural Network (GraphSAGE)
- **Part 5**: Compare methods and top predictions
- **Appendix**: how the test sets were created


In [ ]:
# On Colab, uncomment the following lines to install the required packages
# and to clone the course repository with the data:
# !pip install python-igraph node2vec scikit-learn pandas numpy matplotlib seaborn
# import os, torch
# os.environ['TORCH'] = torch.__version__
# !pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
# !pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
# !pip install -q torch-geometric
# !git clone https://github.com/jgarciab/NetworkScience.git


In [ ]:
# Path to the data folder
# On your computer:
path_data = "../../Data/"
# On Colab uncomment:
# path_data = "/content/NetworkScience/Data/"


In [ ]:
# Core network libraries
import igraph as ig
import networkx as nx  # node2vec expects a NetworkX graph
import graph_tool as gt
from graph_tool.topology import vertex_similarity

# Data and numerics
import numpy as np
import pandas as pd
import itertools
import random

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# ML helpers
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.decomposition import TruncatedSVD

# Embedding/GNN libraries used in Parts 3 and 4
from node2vec import Node2Vec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

pd.options.mode.chained_assignment = None

# Visual style for the whole notebook
custom_params = {
    "axes.spines.right": False, "axes.spines.top": False, "axes.spines.left": False, "axes.spines.bottom": False,
    "lines.linewidth": 2, "grid.color": "lightgray", "legend.frameon": False,
    "xtick.labelcolor": "#484848", "ytick.labelcolor": "#484848",
    "xtick.color": "#484848", "ytick.color": "#484848",
    "text.color": "#484848", "axes.labelcolor": "#484848",
    "axes.titlecolor": "#484848", "figure.figsize": [5, 3],
    "axes.titlelocation": "left",
    "xaxis.labellocation": "left",
    "yaxis.labellocation": "bottom",
}
palette = ["#3d348b", "#e6af2e", "#191716", "#e0e2db"]
sns.set_theme(context='paper', style='white', palette=palette, font_scale=1.3, color_codes=True, rc=custom_params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

# Reproducibility seeds
np.random.seed(1546)
torch.manual_seed(1546)
random.seed(1546)


---
## Introduction: data and prediction target

We use a **Twitter mention network** and a **yeast protein-protein interaction (PPI) network**. Pick one with the `dataset` variable below.

Both datasets have two files:
- `<dataset>_network_prediction.graphml`: the training graph. Some real edges were removed on purpose.
- `<dataset>_network_prediction_test.csv`: the held-out node pairs. `label = 1` means a removed real edge; `label = 0` means a sampled non-edge.

The practical question is: can we use the training graph to guess which held-out pairs are real hidden edges?

The prediction target is a row in a dataframe: two nodes, plus a label saying whether that pair is a real hidden edge. We will hide the labels while making predictions, then reveal them when we evaluate the models.


### Choose a dataset and load the graph

Use `ig.Graph.Read_GraphML` and print basic statistics: number of nodes/edges, density, global clustering (`transitivity_undirected`), and degree assortativity.


In [ ]:
# choose one dataset and load its training graph.
# Small change to try later: change `dataset` from "twitter" to "ppi".
dataset = "twitter"  # options: "twitter", "ppi"

g = ig.Graph.Read_GraphML(f"{path_data}/{dataset}_network_prediction.graphml")
g.simplify()

print(f"Loaded the '{dataset}' graph")
print(f"  nodes (n)     : {g.vcount():,}")
print(f"  edges (m)     : {g.ecount():,}")
print(f"  density       : {g.density():.5f}")
print(f"  transitivity  : {g.transitivity_undirected():.3f}")
print(f"  assortativity : {g.assortativity_degree():.3f}")
print(f"  vertex attrs  : {g.vs.attributes()}")


### Load the held-out test set

The test CSV is tab-separated. The two node columns use the same numeric IDs as the GraphML file. We also attach readable labels, which makes the examples easier to inspect.


In [ ]:
# load the labelled held-out node pairs.
df_test = pd.read_csv(
    f"{path_data}/{dataset}_network_prediction_test.csv",
    sep="\t",
    index_col=0,
)
df_test.columns = [0, 1, "label"]
df_test[[0, 1]] = df_test[[0, 1]].astype(str)
df_test = df_test.drop_duplicates(subset=[0, 1])

id_to_label = dict(zip(g.vs["id"], g.vs["label"]))
df_test["source"] = df_test[0].map(id_to_label)
df_test["target"] = df_test[1].map(id_to_label)

print("Hidden test set:", df_test.shape)
print("  positives:", int((df_test["label"] == 1).sum()))
print("  negatives:", int((df_test["label"] == 0).sum()))
df_test.head()


### Whiteboard challenge: predict 10 held-out pairs

Below are 10 held-out pairs. Look only at the node names and write down which pairs you think are real missing edges.

Later, after each method has made predictions, call `show_whiteboard_predictions("method_name")` to see that method's scores for these same 10 pairs. The labels stay hidden until we explicitly ask to show them.


In [ ]:
whiteboard_pairs = df_test.sample(10, random_state=1546).copy()
whiteboard_pairs["whiteboard_id"] = np.arange(1, len(whiteboard_pairs) + 1)
display(whiteboard_pairs[["whiteboard_id", "source", "target"]].reset_index(drop=True))


### Quick visual check

Plot the training graph once. This is just to see the overall shape of the network.


In [ ]:
# quick shape check, not a final publication plot.
# Small change to try: lower `niter` to 50 for speed, or raise it to 500.
niter = 200
layout = g.layout_fruchterman_reingold(niter=niter)
coords = np.array(layout.coords)

fig, ax = plt.subplots(figsize=(8, 8))
for e in g.es:
    x0, y0 = coords[e.source]
    x1, y1 = coords[e.target]
    ax.plot([x0, x1], [y0, y1], color="lightgray", alpha=0.3, linewidth=0.4, zorder=1)
ax.scatter(coords[:, 0], coords[:, 1], s=5, color=palette[0], zorder=2)
ax.set_axis_off()
ax.set_title(f"{dataset} network (n={g.vcount()}, m={g.ecount()})")
plt.show()


---
## Introduction: candidate-pair dataframe

Most machine-learning tools expect a table: rows are examples, columns are features, and one column is the label we want to learn.

A graph edge list only tells us which edges exist. It does not list all the pairs that *could* have existed but do not. For link prediction we need both:
- positive examples: observed training edges;
- negative examples: node pairs that are not training edges.

So we build one table called `df_pairs`. Each row is one possible non-self pair `(u, v)`. The column `edge` is the training label: `1` means the pair is an observed edge in the training graph, and `0` means it is not.

We also keep `u_idx` and `v_idx`, which are just the row numbers of the two nodes inside the graph object. Node IDs are useful for reading tables; numeric indices are useful for indexing matrices such as `A[u_idx, v_idx]`.

The held-out CSV pairs are separate. We use them only for evaluation.


### Build `df_pairs`

`df_pairs` contains all candidate links we will train on. We leave out self-pairs like `(u, u)` because a node cannot form a link to itself in this practical.


In [ ]:
# Make one row for every possible ordered pair of distinct nodes.
# The main idea: rows are candidate links; `edge=1` marks observed training edges.
node_ids = [str(node_id) for node_id in g.vs["id"]]
node_to_idx = {node_id: i for i, node_id in enumerate(node_ids)}

all_pairs = [
    (u, v)
    for u, v in itertools.product(node_ids, repeat=2)
    if u != v
]
df_pairs = pd.DataFrame(all_pairs, columns=[0, 1])

real_edges = []
for e in g.es:
    a = str(g.vs[e.source]["id"])
    b = str(g.vs[e.target]["id"])
    real_edges.append((a, b))
    real_edges.append((b, a))

df_edges = pd.DataFrame(real_edges, columns=[0, 1]).drop_duplicates()
df_edges["edge"] = 1

df_pairs = df_pairs.merge(df_edges, how="left").fillna({"edge": 0})
df_pairs["edge"] = df_pairs["edge"].astype(int)

# Numeric node indices let us pull pair scores directly from matrices.
df_pairs["u_idx"] = df_pairs[0].map(node_to_idx).astype(int)
df_pairs["v_idx"] = df_pairs[1].map(node_to_idx).astype(int)
pair_u = df_pairs["u_idx"].to_numpy()
pair_v = df_pairs["v_idx"].to_numpy()

print("df_pairs:", df_pairs.shape)
print("positive training pairs:", int(df_pairs["edge"].sum()))
print("negative training pairs:", int((df_pairs["edge"] == 0).sum()))
df_pairs.head()


---
## Introduction: evaluation helper

For each score or feature we will:

1. Add a feature column to `df_pairs`.
2. Train a logistic-regression classifier on the training pairs.
3. Evaluate on the held-out `df_test` pairs.
4. Print the AUC and store the test predictions.

AUC is a ranking score. An AUC of `0.5` is random guessing; an AUC of `1.0` is perfect ranking. You can read it as: if we pick one true hidden edge and one non-edge, how often does the model give the true edge the higher score?

The helper below also saves predictions for the 10 whiteboard pairs. After running any named method, use `show_whiteboard_predictions("method_name")`. Set `show_labels=True` only when we are ready to reveal the answers.


In [ ]:
# Containers filled as we go.
auc_log = {}
test_prediction_log = {}


def create_predictions(df_pairs, df_test,
                       columns=None, return_test=False,
                       solver="lbfgs", name=None, auc_log=None,
                       show_plot=True, verbose=True):
    """
    Train a 5-fold cross-validated logistic regression on df_pairs
    and evaluate it on df_test.

    If `name` is given, the held-out test predictions are saved in
    `test_prediction_log[name]`, so they can be reused for the whiteboard pairs.
    """
    if columns is None:
        non_features = {0, 1, "edge", "u_idx", "v_idx"}
        columns = [c for c in df_pairs.columns if c not in non_features]
    if isinstance(columns, str):
        columns = [columns]

    X_train = df_pairs[columns].values
    y_train = df_pairs["edge"].values

    # Join the held-out pairs to the same feature columns used for training.
    test = pd.merge(df_test, df_pairs, on=[0, 1], how="inner", validate="1:1")

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    model = LogisticRegressionCV(
        cv=5, max_iter=1000, solver=solver,
        class_weight="balanced", n_jobs=1, scoring="roc_auc",
    ).fit(X_train, y_train)

    if verbose:
        print("Model coefficients")
        print(f"  intercept: {model.intercept_[0]: .2f}")
        for v, c in zip(columns, model.coef_[0]):
            print(f"  {v}: {c: .2f}")

    X_test = scaler.transform(test[columns].values)
    y_test = test["label"].values
    y_score = model.predict_proba(X_test)[:, 1]
    y_pred = (y_score >= 0.5).astype(int)

    auc = roc_auc_score(y_test, y_score)

    df_out = test.loc[:, [0, 1, "source", "target", "label"]].copy()
    df_out["pred_prob"] = y_score
    df_out["pred"] = y_pred

    if name is not None:
        test_prediction_log[name] = df_out.copy()

    if show_plot:
        RocCurveDisplay.from_predictions(y_test, y_score)
        plt.plot([0, 1], [0, 1], "--", color="lightgray")
        plt.grid(True)
        plt.title(f"ROC - {name if name else columns}")
        plt.show()

    if verbose:
        print("-" * 50)
        print(f"AUC on the held-out test set: {auc:.3f}")
        print("-" * 50)

    if auc_log is not None and name is not None:
        auc_log[name] = auc

    if return_test:
        return df_out
    return None


def show_whiteboard_predictions(method, show_labels=False, sort_by_probability=True):
    """Show one method's predictions for the 10 whiteboard pairs.

    Use a method name saved by `create_predictions(..., name="method_name")`,
    for example `show_whiteboard_predictions("jaccard")`.
    """
    if isinstance(method, str):
        if method not in test_prediction_log:
            available = sorted(test_prediction_log)
            raise KeyError(
                f"No saved predictions for {method!r}. "
                f"Run that method first, or choose one of: {available}"
            )
        method_name = method
        predictions = test_prediction_log[method].copy()
    else:
        method_name = "custom method"
        predictions = method.copy()

    required = {0, 1, "pred_prob"}
    missing = required - set(predictions.columns)
    if missing:
        raise ValueError(f"Predictions are missing required columns: {sorted(missing)}")

    pred_cols = [0, 1, "pred_prob"]
    if "pred" in predictions.columns:
        pred_cols.append("pred")

    out = whiteboard_pairs[["whiteboard_id", 0, 1, "source", "target", "label"]].merge(
        predictions[pred_cols],
        on=[0, 1],
        how="left",
        validate="1:1",
    )
    out = out.rename(columns={"pred_prob": f"{method_name}_probability"})
    prob_col = f"{method_name}_probability"
    out[prob_col] = out[prob_col].round(3)

    if "pred" in out.columns:
        out[f"{method_name}_prediction"] = np.where(out["pred"] == 1, "edge", "no edge")
    else:
        out[f"{method_name}_prediction"] = np.where(out[prob_col] >= 0.5, "edge", "no edge")

    if show_labels:
        out["true_label"] = np.where(out["label"] == 1, "edge", "no edge")

    if sort_by_probability:
        out = out.sort_values(prob_col, ascending=False)

    display_cols = ["whiteboard_id", "source", "target", prob_col, f"{method_name}_prediction"]
    if show_labels:
        display_cols.append("true_label")

    return out[display_cols].reset_index(drop=True)


---
## Part 1: Local similarity heuristics

Local heuristics look only near the two candidate nodes. The intuition is simple: if two nodes share neighbours, or have similar neighbourhoods, they may be more likely to connect.

Let $\Gamma(u)$ be the neighbours of node $u$, and let $k_u = |\Gamma(u)|$.

The graph-tool API is `vertex_similarity(gt_g, sim_type=..., vertex_pairs=...)`. The names and formulas below follow the graph-tool documentation. These are the available `sim_type` options:

- Dice, `"dice"`: $\frac{2|\Gamma(u) \cap \Gamma(v)|}{|\Gamma(u)| + |\Gamma(v)|}$
- Salton/cosine, `"salton"`: $\frac{|\Gamma(u) \cap \Gamma(v)|}{\sqrt{|\Gamma(u)||\Gamma(v)|}}$
- Hub-promoted, `"hub-promoted"`: $\frac{|\Gamma(u) \cap \Gamma(v)|}{\max(|\Gamma(u)|, |\Gamma(v)|)}$
- Hub-suppressed, `"hub-suppressed"`: $\frac{|\Gamma(u) \cap \Gamma(v)|}{\min(|\Gamma(u)|, |\Gamma(v)|)}$
- Jaccard, `"jaccard"`: $\frac{|\Gamma(u) \cap \Gamma(v)|}{|\Gamma(u) \cup \Gamma(v)|}$
- Adamic-Adar, `"inv-log-weight"`: $\sum_{w \in \Gamma(u) \cap \Gamma(v)} \frac{1}{\log k_w}$
- Resource Allocation, `"resource-allocation"`: $\sum_{w \in \Gamma(u) \cap \Gamma(v)} \frac{1}{k_w}$
- Leicht-Holme-Newman, `"leicht-holme-newman"`: $\frac{|\Gamma(u) \cap \Gamma(v)|}{|\Gamma(u)||\Gamma(v)|}$

Preferential Attachment is not a `vertex_similarity` option, but it is another common local heuristic:

$PA(u,v)=k_u k_v$

For that exercise, use `gt_degree` from the setup cell.

We do not make the shared-neighbour count a separate heuristic block here because $(A^2)_{uv}$ in Part 2 already gives the number of length-2 walks between $u$ and $v$.

We will write the Jaccard code once. Your turn is to choose another option, save it under a column name you like, and compare the AUC.


In [ ]:
# Matrix objects are useful for Parts 2 and 3.
A = np.array(g.get_adjacency().data, dtype=float)
np.fill_diagonal(A, 0)
# Load the same GraphML file directly with graph-tool for fast similarities.
gt_g = gt.load_graph(f"{path_data}/{dataset}_network_prediction.graphml")
gt_g.set_directed(False)

gt_ids = gt_g.vp["_graphml_vertex_id"]
gt_node_to_idx = {str(gt_ids[v]): int(v) for v in gt_g.vertices()}
gt_pairs = np.array([
    (gt_node_to_idx[u], gt_node_to_idx[v])
    for u, v in df_pairs[[0, 1]].to_numpy()
], dtype=int)
gt_degree = gt_g.get_total_degrees(np.arange(gt_g.num_vertices()))

print("graph-tool graph:", gt_g.num_vertices(), "nodes,", gt_g.num_edges(), "edges")
print("candidate pairs:", len(gt_pairs))


### One worked graph-tool example: Jaccard

Jaccard asks: what fraction of the two neighbourhoods is shared?

$J(u,v)=\frac{|\Gamma(u) \cap \Gamma(v)|}{|\Gamma(u) \cup \Gamma(v)|}$

**Change this:** set `sim_type` to another graph-tool option from the list above. If the API name is long, you may also change `score_name` to a shorter label for your notebook table. Then rerun the cell and call `show_whiteboard_predictions(score_name)`.


In [ ]:
sim_type = "jaccard"
score_name = "jaccard"

scores = vertex_similarity(
    gt_g,
    sim_type=sim_type,
    vertex_pairs=gt_pairs,
)
df_pairs[score_name] = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)

create_predictions(
    df_pairs, df_test, [score_name],
    name=score_name, auc_log=auc_log,
)

show_whiteboard_predictions(score_name)


### Part 1 summary: check your AUCs

The table below gives reference AUCs for local heuristic scores. This notebook only computes the option you ran above; the other rows are there so you can check your own modified cell.


In [ ]:
local_reference_auc = pd.DataFrame({
    "Method": [
        "Dice", "Salton/cosine", "Hub-promoted", "Hub-suppressed", "Jaccard",
        "Adamic-Adar", "Resource Allocation", "Leicht-Holme-Newman",
        "Preferential Attachment",
    ],
    "Score column": [
        "dice", "salton", "hub-promoted", "hub-suppressed", "jaccard",
        "inv-log-weight", "resource-allocation", "leicht-holme-newman",
        "preferential_attachment",
    ],
    "twitter reference": [0.894, 0.908, 0.879, 0.900, 0.894, 0.921, 0.926, 0.790, 0.851],
    "ppi reference":     [0.562, 0.562, 0.562, 0.562, 0.562, 0.562, 0.562, 0.562, 0.630],
})

this_run = pd.DataFrame(
    [(m, auc_log[m]) for m in local_reference_auc["Score column"] if m in auc_log],
    columns=["Score column", "This run"],
)
summary = local_reference_auc.merge(this_run, on="Score column", how="left")
summary["This run"] = summary["This run"].round(3)
display(summary.fillna("not run"))


**What to notice.** Local heuristics can be very strong here because the hidden edges were removed from the same network. Many hidden positives still sit in dense neighbourhoods, so a simple structural score can rank them well.


---
## Part 2: Paths and Katz

Local heuristics only look nearby. Path features ask whether two nodes are connected by short walks through the graph.


### Powers of the adjacency matrix

The entry $(A^k)_{uv}$ counts the number of walks of length $k$ from node $u$ to node $v$.

Short walks usually mean stronger evidence than long walks, but trying a few lengths is useful.

**Change this:** after the baseline run, change `path_columns` from `["a2", "a3", "a4"]` to `["a2"]`, then to `["a3"]`, then to `["a2", "a3"]`. Rerun the cell each time and record which walk lengths help most.


In [ ]:
# A^2, A^3 and A^4 count walks of increasing length.
A2 = A @ A
A3 = A2 @ A
A4 = A3 @ A
for M in (A2, A3, A4):
    np.fill_diagonal(M, 0)

df_pairs["a2"] = A2[pair_u, pair_v]
df_pairs["a3"] = A3[pair_u, pair_v]
df_pairs["a4"] = A4[pair_u, pair_v]

# Change this list to compare different walk lengths.
path_columns = ["a2", "a3", "a4"]

create_predictions(
    df_pairs, df_test, path_columns,
    name="paths", auc_log=auc_log,
)


### Katz similarity

Katz adds up walks of all lengths, but longer walks count less:

$S = \sum_{k=1}^{\infty} \beta^k P^k = (I - \beta P)^{-1} - I$

Here $P=D^{-1}A$ is the row-normalized adjacency matrix. Row-normalization makes the scale easy: any $0 < \beta < 1$ is safe, and values closer to `1` give longer walks more influence.

**Change this:** after the baseline run, change `beta` to `0.50`, then to `0.99`. Rerun the cell each time and record how the AUC changes.


In [ ]:
# Katz sums all path lengths, down-weighting longer paths.
def katz_similarity(A, beta=0.95):
    """Closed-form Katz similarity on the row-normalized adjacency matrix."""
    if not 0 < beta < 1:
        raise ValueError("For row-normalized Katz, choose 0 < beta < 1.")

    row_sums = A.sum(axis=1, keepdims=True)
    P = np.divide(A, row_sums, out=np.zeros_like(A, dtype=float), where=row_sums > 0)

    I = np.eye(P.shape[0])
    S = np.linalg.inv(I - beta * P) - I
    print(f"row-normalized Katz with beta = {beta:.2f}")
    return S


beta = 0.95
S_katz = katz_similarity(A, beta=beta)
np.fill_diagonal(S_katz, 0)

df_pairs["katz"] = S_katz[pair_u, pair_v]
create_predictions(
    df_pairs, df_test, ["katz"],
    name="katz", auc_log=auc_log,
)


---
## Part 3: Node embeddings

Embeddings turn each node into a vector. Then a pair of nodes can be compared by the distance between their vectors.

We try two examples: SVD on the adjacency matrix, and node2vec random walks.


### Pairwise vector comparisons

Embeddings give each node a vector. To turn two node vectors into an edge score, we compare the two vectors.

In this notebook, smaller distance means the nodes are more similar. The classifier then learns whether similar vectors tend to mean `edge = 1`.


In [ ]:

toy_vectors = np.array([
    [1.0, 0.0],
    [0.9, 0.1],
    [0.0, 1.0],
])
toy_names = ["A", "B", "C"]

rows = []
for i, j in [(0, 1), (0, 2), (1, 2)]:
    x = toy_vectors[i]
    y = toy_vectors[j]
    cosine_similarity = np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))
    rows.append({
        "pair": f"{toy_names[i]}-{toy_names[j]}",
        "L1 distance": np.abs(x - y).sum(),
        "squared L2 distance": ((x - y) ** 2).sum(),
        "cosine similarity": cosine_similarity,
        "cosine distance": 1 - cosine_similarity,
    })

display(pd.DataFrame(rows).round(3))


In [ ]:
def pairwise_distance(emb, metric="L1"):
    """N x N pairwise distance matrix from row-wise node embeddings.

    metric:
      - 'L1'   : sum |v_i - v_j|     (Manhattan)
      - 'sqL2' : sum (v_i - v_j)^2   (squared Euclidean)
      - 'cos'  : 1 - cos(v_i, v_j)
    """
    if metric == "L1":
        diff = np.abs(emb[:, None] - emb)
        return diff.sum(-1)
    if metric == "sqL2":
        diff = (emb[:, None] - emb) ** 2
        return diff.sum(-1)
    if metric == "cos":
        norms = np.linalg.norm(emb, axis=1, keepdims=True)
        normed = np.divide(emb, norms, out=np.zeros_like(emb, dtype=float), where=norms > 0)
        return 1 - np.einsum("ik,jk->ij", normed, normed)
    raise ValueError(metric)


### Spectral embedding (Truncated SVD)

SVD compresses the adjacency matrix into a smaller number of dimensions:

$A \approx U\Sigma V^T$

Nodes with similar connection patterns should get similar vectors.

**Change this:** after the baseline run, change `n_components` from `10` to `5` or `20`, and change `svd_metric` from `"L1"` to `"sqL2"` or `"cos"`. Rerun the cell and record which combination works best.


In [ ]:
# Spectral embeddings compress each node into a short vector.
# Change n_components and svd_metric, then rerun this cell.
n_components = 10
svd = TruncatedSVD(n_components=n_components, random_state=1546)
svd.fit(A)
emb_svd = svd.components_.T

svd_metric = "L1"  # options: "L1", "sqL2", "cos"
dist_svd = pairwise_distance(emb_svd, metric=svd_metric)
df_pairs["svd"] = dist_svd[pair_u, pair_v]

create_predictions(
    df_pairs, df_test, ["svd"],
    name="svd", auc_log=auc_log,
)


### node2vec

node2vec makes short random walks on the graph. Treat each walk like a sentence and each node like a word. Nodes that appear in similar walk contexts get similar vectors.

Step by step:
1. Convert the igraph training graph directly to NetworkX.
2. Generate biased random walks.
3. Fit skip-gram embeddings from those walks.
4. Extract one vector per node.
5. Compare node vectors with L1 distance.
6. Train the same link-prediction classifier as before.

Key settings:
- **`dimensions`**: vector size.
- **`walk_length`**: how long each walk is.
- **`num_walks`**: how many walks start from each node.
- **`window`**: how many nearby nodes the skip-gram model sees.
- **`p`**: high values discourage immediately walking back.
- **`q`**: low values explore outward; high values stay more local.

**Change this:** after the baseline run, edit `node2vec_params`: try `"q": 0.5` for more outward exploration, then `"q": 2.0` for more local walks. You can also change `dimensions`, `walk_length`, or `window`. Rerun the cell and record both the AUC and how the walk behaviour changed.


In [ ]:
# node2vec turns random walks into node embeddings.
# Change q, dimensions, walk_length, num_walks, or window and rerun this cell.
print("Step 1: convert the igraph training graph to NetworkX for node2vec")
G_nx = g.to_networkx(create_using=nx.Graph)
G_nx = nx.relabel_nodes(
    G_nx,
    {i: g.vs[i]["id"] for i in range(g.vcount())},
    copy=True,
)

node2vec_params = {
    "dimensions": 64,
    "walk_length": 30,
    "num_walks": 10,
    "p": 1.0,
    "q": 1.0,
    "workers": 4,
    "quiet": True,
    "seed": 1546,
}
window = 5

print("Step 2-3: generate biased walks and fit skip-gram embeddings")
n2v = Node2Vec(G_nx, **node2vec_params).fit(
    window=window,
    min_count=1,
    seed=1546,
)

print("Step 4: extract one vector per node")
emb_n2v = np.vstack([n2v.wv.get_vector(node_id) for node_id in g.vs["id"]])

print("Step 5-6: turn node vectors into pair features and evaluate")
dist_n2v = pairwise_distance(emb_n2v, metric="L1")
df_pairs["node2vec"] = dist_n2v[pair_u, pair_v]

create_predictions(
    df_pairs, df_test, ["node2vec"],
    name="node2vec", auc_log=auc_log,
)


---
## Part 4: Graph Neural Network (GraphSAGE)

A GNN learns node vectors while it trains. Each node updates its vector using information from its neighbours.

This part uses the same split as the rest of the practical: message passing and training positives come from `<dataset>_network_prediction.graphml`, and evaluation uses `<dataset>_network_prediction_test.csv` loaded as `df_test`.

A binary edge classifier needs both positive and negative training examples. The positives are the observed GraphML edges. The negatives are sampled from `df_pairs` where `edge = 0`. We sample the same number of negatives as positives so the loss sees a balanced training problem.

Two details matter for undirected link prediction: we exclude both directions of every held-out CSV pair when sampling training negatives, and the edge scorer is symmetric, so `(u, v)` and `(v, u)` get the same kind of representation.

**Change this:** change one setting at a time: `hidden_dim`, `drop_p`, `learning_rate`, or `epochs`. Record the held-out test AUC each time. Notice that dropout is applied after the first GraphSAGE layer, not after the final embedding layer.


In [ ]:
# Use the same GraphML training graph as the rest of the practical.
g_gnn = g
num_nodes = g_gnn.vcount()

train_edges = g_gnn.get_edgelist()
edge_list = train_edges + [(v, u) for (u, v) in train_edges]
edge_index = torch.tensor(np.array(edge_list).T, dtype=torch.long)

# Node features: one-hot identity vectors.
x = torch.eye(num_nodes)
data = Data(x=x, edge_index=edge_index)

train_pos_edge = torch.tensor(np.array(train_edges).T, dtype=torch.long)

print(f"GNN training graph from GraphML: n={num_nodes}, m={len(train_edges)}")
print("edge_index shape:", tuple(edge_index.shape))


In [ ]:
# Use the held-out CSV pairs as the test set.
test_pairs = df_test[[0, 1]].astype(str)
test_pairs_with_idx = test_pairs.merge(
    df_pairs[[0, 1, "u_idx", "v_idx"]],
    on=[0, 1],
    how="left",
    validate="1:1",
)
missing_test_pairs = test_pairs_with_idx[test_pairs_with_idx["u_idx"].isna()]
if not missing_test_pairs.empty:
    raise ValueError("Some held-out CSV pairs are not in df_pairs.")

test_edge = torch.tensor(
    test_pairs_with_idx[["u_idx", "v_idx"]].to_numpy(dtype=int).T,
    dtype=torch.long,
)
test_label = torch.tensor(df_test["label"].to_numpy(dtype=float), dtype=torch.float)

# Sample training negatives, excluding both directions of every held-out CSV pair.
# Otherwise hidden positives could be taught to the GNN as negatives.
held_out_pairs = set(map(tuple, test_pairs.to_numpy()))
held_out_pairs |= {(v, u) for (u, v) in held_out_pairs}

train_negative_pairs = df_pairs.loc[df_pairs["edge"] == 0, [0, 1, "u_idx", "v_idx"]].copy()
keep_mask = [tuple(pair) not in held_out_pairs for pair in train_negative_pairs[[0, 1]].to_numpy()]
train_negative_pairs = train_negative_pairs.loc[keep_mask]
train_negative_pairs = train_negative_pairs.sample(
    n=train_pos_edge.shape[1],
    random_state=1546,
    replace=False,
)
train_neg_edge = torch.tensor(
    train_negative_pairs[["u_idx", "v_idx"]].to_numpy(dtype=int).T,
    dtype=torch.long,
)

print(f"train pos: {train_pos_edge.shape[1]}, train neg: {train_neg_edge.shape[1]}")
print(f"held-out test pairs from CSV: {test_edge.shape[1]}")


In [ ]:
# GraphSAGE encoder + symmetric MLP scorer.
class GraphSAGE(nn.Module):
    """Two-layer GraphSAGE encoder with dropout only after the first layer."""
    def __init__(self, in_dim, hidden_dim, drop_p=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim, aggr="mean")
        self.conv2 = SAGEConv(hidden_dim, hidden_dim, aggr="mean")
        self.dropout = nn.Dropout(p=drop_p)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = self.dropout(h)
        h = self.conv2(h, edge_index)
        return h


class MLPPredictor(nn.Module):
    """Symmetric MLP scorer for undirected edges."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.lin1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, 1)

    def forward(self, h, edge_index):
        h_src = h[edge_index[0]]
        h_dst = h[edge_index[1]]
        pair_features = torch.cat([
            torch.abs(h_src - h_dst),
            h_src * h_dst,
        ], dim=-1)
        hidden = F.relu(self.lin1(pair_features))
        score = self.lin2(hidden)
        return score.view(-1)


def binary_loss(pos_score, neg_score):
    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([torch.ones_like(pos_score), torch.zeros_like(neg_score)])
    return F.binary_cross_entropy_with_logits(scores, labels)


def compute_auc_from_labels(score, labels):
    return roc_auc_score(labels.detach().cpu().numpy(), score.detach().cpu().numpy())


In [ ]:
# Change these settings and rerun the training cell.
hidden_dim = 64
drop_p = 0.3
learning_rate = 0.01
epochs = 80

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = data.to(device)
train_pos_edge = train_pos_edge.to(device)
train_neg_edge = train_neg_edge.to(device)
test_edge = test_edge.to(device)
test_label = test_label.to(device)

model = GraphSAGE(data.num_node_features, hidden_dim=hidden_dim, drop_p=drop_p).to(device)
predictor = MLPPredictor(hidden_dim=hidden_dim).to(device)

optimizer = torch.optim.Adam(
    itertools.chain(model.parameters(), predictor.parameters()),
    lr=learning_rate,
)

print("Training started")
for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    h = model(data.x, train_pos_edge)
    pos_score = predictor(h, train_pos_edge)
    neg_score = predictor(h, train_neg_edge)
    loss = binary_loss(pos_score, neg_score)
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            h = model(data.x, train_pos_edge)
            test_score = predictor(h, test_edge)
            print(f"Epoch {epoch:02d} | Loss {loss.item():.4f} | "
                  f"Held-out AUC {compute_auc_from_labels(test_score, test_label):.4f}")
print("Training complete")


In [ ]:
df_test_gnn = df_test[[0, 1, "source", "target", "label"]].copy()

with torch.no_grad():
    h = model(data.x, train_pos_edge)
    logits = predictor(h, test_edge)
    df_test_gnn["pred_prob"] = torch.sigmoid(logits).cpu().numpy()
    df_test_gnn["pred"] = (df_test_gnn["pred_prob"] >= 0.5).astype(int)

RocCurveDisplay.from_predictions(df_test_gnn["label"], df_test_gnn["pred_prob"])
plt.plot([0, 1], [0, 1], "--", color="lightgray", zorder=0)
plt.title("ROC - GraphSAGE link prediction")
plt.grid(True)
plt.show()

auc_gnn = roc_auc_score(df_test_gnn["label"], df_test_gnn["pred_prob"])
print(f"GNN AUC on the held-out CSV pairs: {auc_gnn:.3f}")
auc_log["gnn"] = auc_gnn
test_prediction_log["gnn"] = df_test_gnn.copy()

show_whiteboard_predictions("gnn")


### Inspect confident false positives

A high-scoring `label = 0` pair is not necessarily a silly prediction. The negative examples are sampled non-edges from the held-out file; some can still look structurally plausible.


In [ ]:
false_pos_gnn = (
    df_test_gnn.query("label == 0")
    .sort_values("pred_prob", ascending=False)
    .head(10)
)
display(false_pos_gnn[["source", "target", "label", "pred_prob", "pred"]])

piratepeel_false_pos = (
    df_test_gnn
    .query("label == 0 and (source == 'PiratePeel' or target == 'PiratePeel')")
    .sort_values("pred_prob", ascending=False)
)

display(piratepeel_false_pos[["source", "target", "label", "pred_prob", "pred"]])


---
## Part 5: Compare methods and top predictions

Each method saved its held-out predictions in `test_prediction_log`. Use this final section to compare AUCs, reveal the 10 whiteboard labels, and inspect the strongest predicted missing edges.

The goal is to ask: which pairs does each method rank highly, and why?


In [ ]:
auc_summary = pd.DataFrame({
    "method": list(auc_log.keys()),
    "held-out AUC": list(auc_log.values()),
}).sort_values("held-out AUC", ascending=False)

auc_summary["held-out AUC"] = auc_summary["held-out AUC"].round(3)
display(auc_summary.reset_index(drop=True))


### Reveal whiteboard labels and inspect top predictions

Choose a saved method with `method_to_inspect`. The default below picks the method with the highest held-out AUC in this run.

**Change this:** set `method_to_inspect` to another method name, such as `"jaccard"`, `"paths"`, `"katz"`, `"node2vec"`, or `"gnn"`. Compare which whiteboard pairs and top held-out pairs change.


First reveal the 10 whiteboard labels for the selected method. Then inspect the highest-scoring held-out pairs across the full test set.


In [ ]:
method_to_inspect = auc_summary.iloc[0]["method"]
n_to_show = 20

print("Selected method:", method_to_inspect)
display(show_whiteboard_predictions(method_to_inspect, show_labels=True))

top_predictions = (
    test_prediction_log[method_to_inspect]
    .sort_values("pred_prob", ascending=False)
    .head(n_to_show)
)

display(top_predictions[["source", "target", "label", "pred_prob", "pred"]])


---
## Student wrap-up

Choose the two changes that moved the AUC the most. Be ready to explain, in plain language, what information those changes added or removed.


---
## Appendix: how the test sets were created

This is documentation, not part of the demo. It shows how the shipped `*_network_prediction.graphml` training graphs and `*_network_prediction_test.csv` files were created from the raw edge lists.

The code is wrapped in `if False:` so it does not run by default.


In [ ]:
def create_network_train_test(df, net_name="twitter"):
    """Hold out 10% of edges as a test set; save the reduced graph + test CSV.

    Mirrors the function used to generate the data shipped with the course,
    but ported to use igraph.
    """
    # Symmetrise and dedup
    df = pd.concat([df, df.rename(columns={0: 1, 1: 0})]).drop_duplicates()

    g = ig.Graph.TupleList(df.itertuples(index=False), directed=False)
    g.simplify()
    g = g.connected_components().giant()

    # Relabel nodes to 0..n-1
    names = g.vs["name"]
    d_conv = {name: i for i, name in enumerate(names)}

    # Adjacency
    A = np.array(g.get_adjacency().data)
    u, v = np.nonzero(A)
    eids = np.random.permutation(len(u))
    test_size = int(len(eids) * 0.1)

    test_pos_u,  test_pos_v  = u[eids[:test_size]], v[eids[:test_size]]
    train_pos_u, train_pos_v = u[eids[test_size:]], v[eids[test_size:]]

    # Negative edges
    adj_neg = A + np.eye(g.vcount()) - 1
    neg_u, neg_v = np.nonzero(adj_neg)
    neg_eids = np.random.choice(len(neg_u), len(u))
    test_neg_u, test_neg_v = neg_u[neg_eids[:test_size]], neg_v[neg_eids[:test_size]]

    pos = pd.DataFrame(np.array([test_pos_u, test_pos_v]).T)
    pos["label"] = 1
    neg = pd.DataFrame(np.array([test_neg_u, test_neg_v]).T)
    neg["label"] = 0
    df_test = pd.concat([pos, neg])
    df_test.to_csv(f"{path_data}{net_name}_network_prediction_test.csv", sep="\t")

    # Remove held-out positives from g
    held = list(zip(test_pos_u, test_pos_v))
    eid_to_drop = g.get_eids(held, error=False)
    eid_to_drop = [e for e in eid_to_drop if e >= 0]
    g.delete_edges(eid_to_drop)

    # Persist the label attribute (original names) on the int-id vertices
    inverse = {i: n for n, i in d_conv.items()}
    g.vs["label"] = [inverse[i] for i in range(g.vcount())]
    g.vs["id"]    = [str(i)     for i in range(g.vcount())]
    g.write_graphml(f"{path_data}{net_name}_network_prediction.graphml")


if False:        # set to True to regenerate the data files
    np.random.seed(1546)
    df = pd.read_csv(f"{path_data}/ic2s2_netsci_3.tsv",
                     sep="\t", usecols=["source", "target"])
    df.columns = [0, 1]
    create_network_train_test(df, net_name="twitter")

    np.random.seed(1546)
    df = pd.read_csv(f"{path_data}/CCSB-Y2H.txt", sep="\t", header=None)
    create_network_train_test(df, net_name="ppi")
